In [ ]:
import os
import base64
import io
from datetime import datetime
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mne
import yasa
import ipywidgets as widgets
from IPython.display import display, HTML
from ipyfilechooser import FileChooser
from scipy.stats import kurtosis as scipy_kurtosis, skew as scipy_skew
from scipy.signal import find_peaks, savgol_filter

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

In [ ]:
def odd(x):
    """Return nearest odd integer to x (required by savgol_filter)."""
    n = int(round(x))
    if n % 2 != 0:
        return n
    return n - 1 if abs(x - (n - 1)) <= abs((n + 1) - x) else n + 1


def drop_suffix_duplicates(raw):
    """Keep only the -0 variant when MNE creates -0/-1 duplicates for repeated channel names."""
    groups = {}
    for ch in raw.ch_names:
        if ch.endswith('-0') or ch.endswith('-1'):
            base = ch.rsplit('-', 1)[0]
            groups.setdefault(base, []).append(ch)
    to_drop = []
    for base, ch_list in groups.items():
        if any(c.endswith('-0') for c in ch_list):
            to_drop.extend(c for c in ch_list if not c.endswith('-0'))
    if to_drop:
        raw.drop_channels(to_drop)
    return raw, to_drop


def adapt_remap_dict_to_suffixes(raw, remap_dict):
    """Handle MNE's -0 suffix when the remap config uses the base channel name."""
    ch_set = set(raw.ch_names)
    new_remap = {}
    for base_label, target in remap_dict.items():
        if base_label in ch_set:
            new_remap[base_label] = target
        elif f'{base_label}-0' in ch_set:
            new_remap[f'{base_label}-0'] = target
    return new_remap


def get_phys_bounds_uV(extras, ch_idx):
    """
    Reconstruct physical_min/max (in uV) from MNE _raw_extras.
    MNE 1.9 stores physical_max but not physical_min explicitly.
    For symmetric 16-bit EDF: physical_min = -physical_max + 2*offset.
    Returns (phys_min_uV, phys_max_uV).
    """
    phys_max = float(extras['physical_max'][ch_idx])
    offset = float(extras['offsets'][ch_idx])
    return -phys_max + 2 * offset, phys_max


def compute_signal_metrics(sig_uV, phys_min_uV, phys_max_uV):
    """Compute all quality metrics for one EEG channel (unfiltered signal, in uV)."""
    cur_std = float(np.std(sig_uV))
    u = np.unique(sig_uV)

    # Flat signal: fraction of consecutive sample pairs with no change.
    # Tolerance = 2x ADC resolution to handle rounding, minimum 0.06 uV.
    if len(u) > 1:
        res = float(np.min(np.abs(np.diff(u))))
        flat_atol = max(2 * res, 0.06)
    else:
        flat_atol = 0.06
    flat_pct = float(np.isclose(np.diff(sig_uV), 0, atol=flat_atol, rtol=0.0).mean()) * 100

    # EDF physical bounds: fraction of samples at/near the header declared limits.
    # Detects saturation at the EDF dynamic range boundary (hard clipping).
    if not (np.isnan(phys_min_uV) or np.isnan(phys_max_uV)):
        bounds_pct = float(
            ((sig_uV <= phys_min_uV + 0.5) | (sig_uV >= phys_max_uV - 0.5)).mean()
        ) * 100
    else:
        bounds_pct = 0.0

    # Histogram + Savitzky-Golay smoothing + peak detection.
    # SG window = 2% of bin count (at least 5) to smooth noise without flattening real peaks.
    # Peak prominence threshold = 25% of the histogram maximum to ignore minor bumps.
    if cur_std < 1e-10 or len(u) < 3:
        histo, edges = np.histogram(sig_uV, bins=10)
        sg_window = 5
    else:
        histo, edges = np.histogram(sig_uV, bins='scott')
        sg_window = max(5, odd(0.02 * len(histo)))
    bin_centers = (edges[:-1] + edges[1:]) / 2
    y_smooth = savgol_filter(histo.astype(float), window_length=sg_window, polyorder=3)
    prom_thresh = 0.25 * float(np.max(y_smooth)) if np.max(y_smooth) > 0 else 0.0
    peaks, _ = find_peaks(y_smooth, prominence=prom_thresh)

    # Extreme histogram bins: fraction of samples in the outermost bins.
    # Proxy for in-range clipping (signal saturating within the declared EDF physical range).
    total = float(np.sum(histo))
    hist_extreme_pct = float((histo[0] + histo[-1]) / total * 100) if total > 0 else 0.0

    # Percentile-based amplitude metrics: robust characterization of the signal tail.
    # p99 / p99.9 of |amplitude| capture sustained artifacts (movement bursts, electrode pops)
    # without being dominated by a single sample like the max would be.
    abs_uV = np.abs(sig_uV)

    # Distribution statistics.
    # Fischer kurtosis: 0 = normal; >0 = heavy tails/spikes; <0 = flat/platykurtic.
    return {
        'mean_uV': float(np.mean(sig_uV)),
        'std_uV': cur_std,
        'kurtosis': float(scipy_kurtosis(sig_uV, fisher=True)),
        'skewness': float(scipy_skew(sig_uV)),
        'p99_abs_uV': float(np.percentile(abs_uV, 99.0)),
        'p999_abs_uV': float(np.percentile(abs_uV, 99.9)),
        'flat_pct': flat_pct,
        'bounds_pct': bounds_pct,
        'hist_extreme_pct': hist_extreme_pct,
        'n_peaks': int(len(peaks)),
        'histo': histo,
        'edges': edges,
        'bin_centers': bin_centers,
        'y_smooth': y_smooth,
        'peaks': peaks,
    }


def flag_channel(metrics, thresholds):
    """Return list of flag reasons for a channel, or empty list if within all thresholds."""
    reasons = []
    if metrics['flat_pct'] > thresholds['flat_pct']:
        reasons.append(f"flat_pct={metrics['flat_pct']:.1f}% > {thresholds['flat_pct']}%")
    if metrics['bounds_pct'] > thresholds['bounds_pct']:
        reasons.append(f"bounds_pct={metrics['bounds_pct']:.2f}% > {thresholds['bounds_pct']}%")
    if metrics['n_peaks'] >= thresholds['n_peaks']:
        shape = 'bimodal' if metrics['n_peaks'] == 2 else 'multimodal'
        reasons.append(f"n_peaks={metrics['n_peaks']} ({shape})")
    # Low std detects dead or near-dead channels (normal EEG std >> 5 µV).
    # Kurtosis was removed: EEG PSG has physiologically high kurtosis (spindles, K-complexes)
    # causing too many false positives (baseline Fp1 kurtosis ~466).
    if metrics['std_uV'] < thresholds['std_low']:
        reasons.append(f"std={metrics['std_uV']:.2f} µV < {thresholds['std_low']} µV (dead/flat channel)")
    # Extreme histogram bins proxy for in-range clipping (signal saturating within
    # the declared EDF physical range, undetectable via bounds_pct alone).
    if metrics['hist_extreme_pct'] > thresholds['hist_extreme_pct']:
        reasons.append(f"hist_extreme={metrics['hist_extreme_pct']:.2f}% > {thresholds['hist_extreme_pct']}%")
    return reasons


def plot_histogram_figure(metrics, ch_name, y_max=None, x_lim=None):
    """Return a matplotlib figure with the signal amplitude distribution."""
    edges = metrics['edges']
    histo = metrics['histo']
    bin_centers = metrics['bin_centers']
    y_smooth = metrics['y_smooth']
    peaks = metrics['peaks']
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(edges[:-1], histo, width=edges[1] - edges[0], align='edge',
           color='#d4e8f5', edgecolor='#7ab4d0')
    ax.plot(bin_centers, y_smooth, color='#6a51a3', linewidth=2,
            label='Savitzky-Golay smooth')
    ax.plot(bin_centers[peaks], y_smooth[peaks], 'r+', markersize=10, markeredgewidth=2,
            label=f'{len(peaks)} peak(s) detected')
    ax.set_xlabel('Amplitude (μV)', fontsize=11)
    ax.set_ylabel('Counts', fontsize=11)
    if y_max is not None:
        ax.set_ylim(0, y_max * 1.05)
    if x_lim is not None:
        ax.set_xlim(-x_lim, x_lim)
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    ax.legend(fontsize=9)
    plt.tight_layout()
    return fig


def plot_timeseries_figure(sig_uV, sf, y_lim):
    """Raw time series (downsampled to ~10 Hz for display), fixed Y-axis."""
    step = max(1, int(sf / 10))
    t_h = np.arange(0, len(sig_uV), step) / sf / 3600
    fig, ax = plt.subplots(figsize=(10, 2))
    ax.plot(t_h, sig_uV[::step], linewidth=0.3, color='#2a6099', rasterized=True)
    ax.set_xlim(0, len(sig_uV) / sf / 3600)
    ax.set_ylim(-y_lim, y_lim)
    ax.set_xlabel('Time (h)', fontsize=11)
    ax.set_ylabel('µV', fontsize=11)
    ax.axhline(0, color='#bbb', linewidth=0.5, linestyle='--')
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    return fig


# Per-metric interpretation shown as a third column in the metrics table.
METRIC_INTERPRETATIONS = {
    'Kurtosis (Fischer)': 'PSG EEG: typically ~100&ndash;300; compare across channels &mdash; a channel significantly lower than the others&nbsp;=&nbsp;suspicious',
    'Skewness': '~0&nbsp;=&nbsp;symmetric; |skew|&nbsp;&gt;&nbsp;1&nbsp;=&nbsp;notable asymmetry',
    'n_peaks (distribution)': '1&nbsp;=&nbsp;unimodal; 2&nbsp;=&nbsp;bimodal (DC drift); &ge;3&nbsp;=&nbsp;coarse quantization',
    'p99 |amplitude|': '99% of samples are below this value&nbsp;; compare across channels&nbsp;',
    'p99.9 |amplitude|': '99.9% of samples are below this value&nbsp;; compare across channels&nbsp;',
}

def _fig_to_b64(fig):
    """Render a matplotlib figure to a base64-encoded PNG for HTML embedding."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)
    return b64


def generate_dataset_overview(df, reports_root):
    """
    Generate dataset_overview.html from a quality_summary DataFrame.

    Global section:
      - stats table (all electrodes pooled)
      - n_peaks frequency table by electrode
      - pooled distribution plot (all electrodes combined, one box per metric)
      - grouped comparison plot (one box per metric per electrode)

    Per-electrode sections:
      - stats table for that electrode
      - distribution plot across participants (scatter colored red=flagged, grey=ok)

    Regenerated each run from the full cumulative quality_summary.tsv.
    """
    KEY_METRICS = [
        "std_uV", "flat_pct", "bounds_pct", "hist_extreme_pct",
        "p99_abs_uV", "p999_abs_uV",
    ]
    ALL_NUMERIC = [
        "mean_uV", "std_uV", "kurtosis", "skewness",
        "p99_abs_uV", "p999_abs_uV",
        "flat_pct", "bounds_pct", "hist_extreme_pct",
    ]
    LABELS = {
        "std_uV": "Std dev (µV)",
        "flat_pct": "Flat signal (%)",
        "bounds_pct": "At EDF bounds (%)",
        "hist_extreme_pct": "Extreme histogram (%)",
        "p99_abs_uV": "p99 |amplitude| (µV)",
        "p999_abs_uV": "p99.9 |amplitude| (µV)",
        "mean_uV": "Mean (µV)",
        "kurtosis": "Kurtosis (Fischer)",
        "skewness": "Skewness",
    }

    channels = sorted(df["channel"].unique())
    n_participants = int(df["file_id"].nunique())
    n_entries = len(df)
    n_flagged = int(df["exclude"].sum()) if "exclude" in df.columns else "?"
    n_flagged_part = int(df.groupby("file_id")["exclude"].any().sum()) if "exclude" in df.columns else "?"

    CSS = (
        "<style>"
        "body{font-family:Arial,sans-serif;max-width:1100px;margin:0 auto;padding:20px;color:#333}"
        "h1{color:#333;border-bottom:2px solid #ccc;padding-bottom:8px}"
        "h2{color:#333;margin-top:32px;border-bottom:1px solid #ddd;padding-bottom:4px}"
        "h3{color:#333;margin-top:16px}"
        "table{border-collapse:collapse;width:100%;font-size:.9em;margin-bottom:20px}"
        "th{background:#888;color:#fff;padding:6px 12px;text-align:left}"
        "td{padding:5px 12px;border-bottom:1px solid #eee}"
        "tr:nth-child(even){background:#f8f8f8}"
        ".flag{color:#c0392b;font-weight:bold}"
        "img{max-width:100%;margin:10px 0}"
        ".footer{color:#888;font-size:.8em;margin-top:40px;"
        "border-top:1px solid #ddd;padding-top:10px}"
        "</style>"
    )

    def _stats_table(data):
        HDR = (
            "<tr><th>Metric</th><th>Mean</th><th>Median</th>"
            "<th>p5</th><th>p25</th><th>p75</th><th>p95</th></tr>"
        )
        rows = []
        for m in ALL_NUMERIC:
            s = data[m].dropna() if m in data.columns else pd.Series(dtype=float)
            if s.empty:
                continue
            rows.append(
                f"<tr><td>{LABELS.get(m, m)}</td>"
                f"<td>{s.mean():.3g}</td><td>{s.median():.3g}</td>"
                f"<td>{s.quantile(0.05):.3g}</td><td>{s.quantile(0.25):.3g}</td>"
                f"<td>{s.quantile(0.75):.3g}</td><td>{s.quantile(0.95):.3g}</td></tr>"
            )
        return f"<table>{HDR}{''.join(rows)}</table>"

    def _npeaks_table(data):
        if "n_peaks" not in data.columns:
            return ""
        HDR = (
            "<tr><th>Electrode</th><th>1 peak (normal)</th>"
            "<th>2 peaks (DC drift)</th><th>≥3 peaks (quantization)</th></tr>"
        )
        rows = []
        for ch in sorted(data["channel"].unique()):
            s = data[data["channel"] == ch]["n_peaks"]
            c1 = int((s == 1).sum())
            c2 = int((s == 2).sum())
            c3 = int((s >= 3).sum())
            f2 = ' class="flag"' if c2 else ""
            f3 = ' class="flag"' if c3 else ""
            rows.append(
                f"<tr><td>{ch}</td><td>{c1}</td>"
                f"<td{f2}>{c2}</td><td{f3}>{c3}</td></tr>"
            )
        return (
            "<h3>n_peaks distribution by electrode</h3>"
            f"<table>{HDR}{''.join(rows)}</table>"
        )

    def _scatter_with_flag(ax, x_center, vals, flags, jitter_width=0.07):
        """Scatter individual data points colored by flag status.
        Red = channel flagged for this participant; grey = within thresholds.
        showfliers=False must be set on the boxplot to avoid duplicate markers.
        """
        colors = ["#c0392b" if f else "#555555" for f in flags]
        jitter = np.random.uniform(-jitter_width, jitter_width, len(vals))
        for v, j, c in zip(vals, jitter, colors):
            ax.scatter(x_center + j, v, color=c, s=28, alpha=0.75, zorder=3,
                       linewidths=0)

    def _flag_legend(fig):
        """Add a flag/ok legend to the figure."""
        handles = [
            matplotlib.lines.Line2D(
                [0], [0], marker="o", color="w", markerfacecolor="#c0392b",
                markersize=8, label="Flagged",
            ),
            matplotlib.lines.Line2D(
                [0], [0], marker="o", color="w", markerfacecolor="#555555",
                markersize=8, label="Not flagged",
            ),
        ]
        fig.legend(handles=handles, loc="upper right", fontsize=9, framealpha=0.9)

    def _vals_and_flags(data, metric):
        """Return aligned (values, flags) arrays for rows where metric is not NaN."""
        valid = data[metric].notna()
        vals = data.loc[valid, metric].values
        if "exclude" in data.columns:
            flags = data.loc[valid, "exclude"].fillna(False).astype(bool).values
        else:
            flags = np.zeros(len(vals), dtype=bool)
        return vals, flags

    def _pooled_boxplots(data):
        """All electrodes combined — one box per metric.
        Gives the dataset-wide distribution before the per-electrode comparison.
        """
        ncols = min(3, len(KEY_METRICS))
        nrows = int(np.ceil(len(KEY_METRICS) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)
        axes = axes.flatten()
        for ax, m in zip(axes, KEY_METRICS):
            vals, flags = _vals_and_flags(data, m)
            if len(vals) == 0:
                ax.set_visible(False)
                continue
            bp = ax.boxplot(
                [vals], patch_artist=True, showfliers=False,
                medianprops=dict(color="#c0392b", linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2),
            )
            bp["boxes"][0].set_facecolor("#d4e8f5")
            bp["boxes"][0].set_alpha(0.8)
            _scatter_with_flag(ax, x_center=1, vals=vals, flags=flags)
            ax.set_title(LABELS.get(m, m), fontsize=11, fontweight="bold")
            ax.set_xticks([])
            for sp in ["right", "top"]:
                ax.spines[sp].set_visible(False)
        for ax in axes[len(KEY_METRICS):]:
            ax.set_visible(False)
        _flag_legend(fig)
        fig.suptitle(
            f"All electrodes pooled — metric distributions (n={len(data)} channel records)",
            fontsize=12, y=1.02,
        )
        plt.tight_layout()
        return f'<img src="data:image/png;base64,{_fig_to_b64(fig)}" style="max-width:100%;">'

    def _grouped_boxplots(data):
        """One subplot per metric; x-axis = electrode — compares electrodes side by side.
        showfliers=False keeps scale anchored to the IQR range (no outlier points
        inflating the axis). Individual values are not shown here; see per-electrode
        sections for participant-level detail.
        """
        ncols = min(3, len(KEY_METRICS))
        nrows = int(np.ceil(len(KEY_METRICS) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)
        axes = axes.flatten()
        for ax, m in zip(axes, KEY_METRICS):
            groups = [data[data["channel"] == ch][m].dropna().values for ch in channels]
            bp = ax.boxplot(
                groups, labels=channels, patch_artist=True, showfliers=False,
                medianprops=dict(color="#c0392b", linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2),
            )
            for patch in bp["boxes"]:
                patch.set_facecolor("#d4e8f5")
                patch.set_alpha(0.8)
            ax.set_title(LABELS.get(m, m), fontsize=11, fontweight="bold")
            rot = 30 if len(channels) > 4 else 0
            ax.tick_params(axis="x", labelrotation=rot, labelsize=9)
            for sp in ["right", "top"]:
                ax.spines[sp].set_visible(False)
        for ax in axes[len(KEY_METRICS):]:
            ax.set_visible(False)
        plt.tight_layout()
        return f'<img src="data:image/png;base64,{_fig_to_b64(fig)}" style="max-width:100%;">'

    def _electrode_boxplots(ch_data, ch_name):
        """One subplot per metric; distribution of that metric across participants
        for this electrode. Each dot = one participant.
        Red dot = this channel was flagged for that participant.
        Grey dot = within all thresholds.
        showfliers=False removes matplotlib's default outlier markers so each
        participant appears exactly once as a colored dot.
        """
        ncols = min(3, len(KEY_METRICS))
        nrows = int(np.ceil(len(KEY_METRICS) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)
        axes = axes.flatten()
        for ax, m in zip(axes, KEY_METRICS):
            vals, flags = _vals_and_flags(ch_data, m)
            if len(vals) == 0:
                ax.set_visible(False)
                continue
            bp = ax.boxplot(
                [vals], patch_artist=True, showfliers=False,
                medianprops=dict(color="#c0392b", linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2),
            )
            bp["boxes"][0].set_facecolor("#d4e8f5")
            bp["boxes"][0].set_alpha(0.8)
            _scatter_with_flag(ax, x_center=1, vals=vals, flags=flags)
            ax.set_title(LABELS.get(m, m), fontsize=11, fontweight="bold")
            ax.set_xticks([])
            for sp in ["right", "top"]:
                ax.spines[sp].set_visible(False)
        for ax in axes[len(KEY_METRICS):]:
            ax.set_visible(False)
        _flag_legend(fig)
        fig.suptitle(
            f"Electrode {ch_name} — distribution across {len(ch_data)} participants",
            fontsize=12, y=1.02,
        )
        plt.tight_layout()
        return f'<img src="data:image/png;base64,{_fig_to_b64(fig)}" style="max-width:100%;">'

    # Build global section
    global_block = (
        "<h2>All electrodes — pooled statistics</h2>"
        f"<h3>Summary statistics (all channels combined, n={n_entries})</h3>"
        + _stats_table(df)
        + _npeaks_table(df)
        + "<h3>Pooled distribution of each metric (all electrodes combined)</h3>"
        + _pooled_boxplots(df)
        + "<h3>Distributions by electrode (comparison)</h3>"
        + _grouped_boxplots(df)
    )

    # Build per-electrode sections
    electrode_blocks = ""
    for ch in channels:
        ch_data = df[df["channel"] == ch]
        electrode_blocks += (
            f"<h2>Electrode: {ch}</h2>"
            f"<h3>Statistics across {len(ch_data)} participants</h3>"
            + _stats_table(ch_data)
            + _electrode_boxplots(ch_data, ch)
        )

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    html = (
        '<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">'
        "<title>Dataset Quality Overview</title>"
        f"{CSS}</head><body>"
        "<h1>Dataset Quality Overview</h1>"
        f"<p><b>Dataset:</b> {reports_root.parent.name}&nbsp;|&nbsp;"
        f"<b>Participants:</b> {n_participants}&nbsp;|&nbsp;"
        f"<b>Entries (channel × participant):</b> {n_entries}&nbsp;|&nbsp;"
        f"<b>Flagged channels:</b> {n_flagged}&nbsp;|&nbsp;"
        f"<b>Participants with ≥1 flag:</b> {n_flagged_part}</p>"
        + global_block
        + electrode_blocks
        + '<p class="footer">'
        f"Generated by quality_overview_voila.ipynb — {timestamp} — "
        f"Source: quality_summary.tsv ({n_entries} entries, {n_participants} participants)"
        "</p></body></html>"
    )

    (reports_root / "dataset_overview.html").write_text(html, encoding="utf-8")

## Preprocessing &mdash; Phase 1: Quality Overview 

This tool computes **signal quality metrics per channel** for every EDF file in a folder. It produces:
- One **HTML report** per participant (histograms, time series, descriptive stats, hypnograms/spectrograms)
- A **`quality_summary.tsv`** with all numeric metrics
- A **`dataset_overview.html`** with dataset-level statistics and distributions per electrode

**Instructions:**
1. Use the first file chooser to select your **data folder**.
2. Select the **remap/reref config JSON** produced by *select&amp;remap_channels_edf*.
3. Adjust the **hypnogram suffix** if needed and the **quality thresholds**.
4. Click **Run Analysis**.

Quality metrics are computed on the **raw, unfiltered signal**. The hypnospectrogram uses a 0.1&ndash;40&nbsp;Hz bandpass filter (applied only for the plot).

In [ ]:
fc_folder = FileChooser()
fc_folder.title = '<b>Select your data folder:</b>'
fc_folder.show_only_dirs = True

hypno_suffix = widgets.Text(
    value='_Hypnogram_remapped.txt',
    description='Hypno suffix:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
hypno_suffix_info = widgets.HTML(value='')

existing_reports_info = widgets.HTML(value='')

fc_config = FileChooser(filter_pattern='*.json')
fc_config.title = '<b>Select remap/reref config JSON</b> (from select&amp;remap_channels_edf):'

def update_existing_reports_info(chooser):
    if not chooser.selected:
        existing_reports_info.value = ''
        hypno_suffix_info.value = ''
        return
    data_folder = Path(chooser.selected)
    edf_files = sorted(data_folder.rglob('*.edf'))
    n_total = len(edf_files)
    if n_total == 0:
        existing_reports_info.value = '<small style="color:#888;">No EDF files found in selected folder (recursive scan).</small>'
        hypno_suffix_info.value = ''
        return
    reports_root = data_folder / 'reports_quality_overview'
    n_existing = sum(
        1 for f in edf_files
        if (reports_root / f.parent.relative_to(data_folder) / f'{f.stem}_quality_overview.html').exists()
    )
    existing_reports_info.value = (
        f'<small style="color:#555;">'
        f'{n_existing} / {n_total} participant(s) already have an existing report.</small>'
    )
    # --- Hypnogram suffix detection ---
    # For each EDF, count all .txt files whose name starts with the EDF stem (no break:
    # a participant can have both a raw and a remapped hypno).
    # Selection: among suffixes appearing for >=50% of the maximum count, prefer the
    # longest (more specific = remapped/processed version). The 50% threshold prevents
    # rare/accidental long suffixes from winning over the true candidate.
    # Tiebreaker when two candidates are equally long: highest count.
    all_txt = list(data_folder.rglob('*.txt'))
    suffix_counts = {}
    for edf in edf_files:
        for txt in all_txt:
            if txt.name.startswith(edf.stem):
                suffix = txt.name[len(edf.stem):]
                suffix_counts[suffix] = suffix_counts.get(suffix, 0) + 1
    if not suffix_counts:
        hypno_suffix_info.value = (
            '<small style="color:#e67e00;">No hypnogram files detected — '
            'check that hypnograms are present or adjust the suffix manually.</small>'
        )
    else:
        max_count = max(suffix_counts.values())
        candidates = {s: c for s, c in suffix_counts.items() if c >= max_count * 0.5}
        best_suffix, best_count = max(candidates.items(), key=lambda x: (len(x[0]), x[1]))
        hypno_suffix.value = best_suffix
        parts = [f'<b>{s}</b>&nbsp;(×{c})' for s, c in sorted(suffix_counts.items(), key=lambda x: -x[1])]
        color = '#2e7d32' if best_count == n_total else '#e67e00'
        hypno_suffix_info.value = (
            f'<small style="color:{color};">Detected:&nbsp;{"&nbsp;·&nbsp;".join(parts)}'
            f'&nbsp;— {best_count}/{n_total} EDF files matched</small>'
        )

fc_folder.register_callback(update_existing_reports_info)

skip_existing = widgets.Checkbox(
    value=True,
    description='Skip participants with an existing report',
    style={'description_width': 'initial'}
)

thresh_flat = widgets.BoundedFloatText(
    value=3.5, min=0.0, max=100.0, step=0.5,
    description='flat_pct (%) >',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)
thresh_bounds = widgets.BoundedFloatText(
    value=1.0, min=0.0, max=100.0, step=0.1,
    description='bounds_pct (%) >',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)
thresh_peaks = widgets.BoundedIntText(
    value=2, min=2, max=10, step=1,
    description='n_peaks >=',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)
thresh_std_low = widgets.BoundedFloatText(
    value=5.0, min=0.0, max=500.0, step=0.5,
    description='std_uV (µV) <',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)
thresh_hist_extreme = widgets.BoundedFloatText(
    value=1.0, min=0.0, max=100.0, step=0.1,
    description='hist_extreme_pct (%) >',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)


def thresh_row(w, desc):
    return widgets.HBox([
        w,
        widgets.HTML(
            f'<small style="color:#555;margin-left:12px;line-height:32px;">{desc}</small>'
        )
    ])


display(
    HTML('<h3 style="margin-top:4px;">&#128193; Data</h3>'),
    fc_folder,
    hypno_suffix,
    hypno_suffix_info,
    fc_config,
    existing_reports_info,
    skip_existing,
    HTML(
        '<h3 style="margin-top:20px;">&#9881; Quality thresholds</h3>'
        '<small style="color:#555;">Channels exceeding any threshold will be flagged '
        'in the report and in <em>dataset_overview.html</em>.</small>'
    ),
    widgets.VBox([
        thresh_row(thresh_flat,
                   'Fraction of consecutive equal samples — detects flat segments and dead channels'),
        thresh_row(thresh_bounds,
                   'Fraction at EDF physical-range limits — detects hard saturation (declared range)'),
        thresh_row(thresh_peaks,
                   'Peaks in amplitude distribution — ≥2: DC drift (bimodal); ≥3: coarse quantization'),
        thresh_row(thresh_std_low,
                   'Signal std dev — very low std (< 5 µV) flags a dead or disconnected channel'),
        thresh_row(thresh_hist_extreme,
                   'Fraction in outermost histogram bins — detects in-range clipping (within EDF range)'),
    ])
)

In [ ]:
btn_run = widgets.Button(
    description='▶  Run Analysis',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)
progress = widgets.IntProgress(
    min=0, max=1, value=0, bar_style='info',
    layout=widgets.Layout(width='500px')
)
progress_label = widgets.Label(value='')
out = widgets.Output()


def run_analysis(btn):
    btn.disabled = True
    out.clear_output()
    try:

        # --- Validate inputs ---
        if not fc_folder.selected:
            with out:
                print('ERROR: Please select a data folder.')
            return
        if not fc_config.selected:
            with out:
                print('ERROR: Please select the remap/reref config JSON.')
            return

        data_folder = Path(fc_folder.selected)
        config_path = Path(fc_config.selected)
        reports_root = data_folder / 'reports_quality_overview'
        reports_root.mkdir(exist_ok=True)

        try:
            with open(config_path, 'r', encoding='utf-8') as f:
                config_dict = json.load(f)
        except Exception as e:
            with out:
                print(f'ERROR loading config: {e}')
            return

        edf_files = sorted(data_folder.rglob('*.edf'))
        if not edf_files:
            with out:
                print(f'No EDF files found in {data_folder}')
            return

        progress.max = len(edf_files)
        progress.value = 0
        hypno_lookup = {p.name: p for p in data_folder.rglob('*.txt')}

        thresholds = {
            'flat_pct': thresh_flat.value,
            'bounds_pct': thresh_bounds.value,
            'n_peaks': thresh_peaks.value,
            'std_low': thresh_std_low.value,
            'hist_extreme_pct': thresh_hist_extreme.value,
        }
        # Build per-report interpretations: threshold-based metrics show the actual
        # threshold value used for this run (set manually in the notebook widgets).
        interpretations = dict(METRIC_INTERPRETATIONS)
        interpretations.update({
            'Flat signal (%)': f'flag if&nbsp;&gt;&nbsp;{thresholds["flat_pct"]:g}%&nbsp;&mdash; threshold defined manually in the notebook ; typical threshold is ??',
            'At EDF bounds (%)': f'flag if&nbsp;&gt;&nbsp;{thresholds["bounds_pct"]:g}%&nbsp;&mdash; threshold defined manually in the notebook ; typical threshold is ??',
            'Extreme histogram (%)': f'flag if&nbsp;&gt;&nbsp;{thresholds["hist_extreme_pct"]:g}%&nbsp;&mdash; threshold defined manually in the notebook ; typical threshold is ??',
        })

        rows_summary = []
        failed = []
        attempted_ids = set()  # file_ids actually processed (not skipped)
        t_start = time.time()

        for i, edf_path in enumerate(edf_files):
            file_id = edf_path.stem
            progress.value = i
            progress_label.value = f'{file_id}  ({i + 1}/{len(edf_files)})'

            # Mirror the EDF subfolder structure under reports_root.
            edf_relative = edf_path.parent.relative_to(data_folder)
            edf_reports_dir = reports_root / edf_relative
            report_path = edf_reports_dir / f'{file_id}_quality_overview.html'

            if skip_existing.value and report_path.exists():
                with out:
                    print(f'Skipped (report exists): {file_id}')
                continue

            attempted_ids.add(file_id)
            edf_reports_dir.mkdir(parents=True, exist_ok=True)

            # --- Load hypnogram ---
            _hypno_name = f'{file_id}{hypno_suffix.value}'
            hypno_path = hypno_lookup.get(_hypno_name, edf_path.parent / _hypno_name)
            hypno_vec = None
            hypno_warning = None
            hypno_warning_plain = None
            if not hypno_path.exists():
                hypno_warning = (
                    f'Hypnogram not found: <b>{hypno_path.name}</b>. '
                    f'Spectrogram skipped. Possible causes: '
                    f'(1) the <i>remap_hypno_voila</i> notebook was not run or did not complete; '
                    f'(2) the hypnogram suffix above does not match the actual file suffix.'
                )
                hypno_warning_plain = f'Hypnogram not found: {hypno_path.name} (spectrogram skipped): check that the suffix is correct or that remap_hypno_voila was run.'
            else:
                try:
                    expert_hypno = np.loadtxt(str(hypno_path), dtype=str).astype('<U10')
                    stage_map = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
                    hypno_vec = np.full(len(expert_hypno), np.nan)
                    for stage, val in stage_map.items():
                        hypno_vec[expert_hypno == stage] = val
                    # MT (movement time) is tolerated: YASA treats it as artifact (NaN), no warning needed.
                    # Any other unrecognised label means the hypnogram was not converted to AASM convention.
                    _tolerated = {'MT'}
                    _unknown_mask = np.array([s not in stage_map and s not in _tolerated for s in expert_hypno])
                    _unknown_labels = sorted(set(expert_hypno[_unknown_mask]))
                    _n_unknown = int(_unknown_mask.sum())
                    _pct_unknown = _n_unknown / len(expert_hypno) * 100
                    if _pct_unknown > 10:
                        hypno_vec = None
                        hypno_warning = (
                            f'Hypnogram labels not AASM: {_n_unknown}/{len(expert_hypno)} epochs '
                            f'({_pct_unknown:.1f}%) have unrecognised labels '
                            f'({", ".join(_unknown_labels)}). Spectrogram skipped. '
                            f'Run <i>remap_hypno_voila</i> to convert labels to W/N1/N2/N3/R.'
                        )
                        hypno_warning_plain = (
                            f'Hypnogram labels not AASM: {_n_unknown}/{len(expert_hypno)} epochs '
                            f'({_pct_unknown:.1f}%) have unrecognised labels '
                            f'({", ".join(_unknown_labels)}). Spectrogram skipped.'
                        )
                    elif _unknown_labels:
                        hypno_warning = (
                            f'{_n_unknown} epoch(s) ({_pct_unknown:.1f}%) have unrecognised label(s): '
                            f'{", ".join(_unknown_labels)}. These epochs will be treated as artifact '
                            f'in the spectrogram.'
                        )
                        hypno_warning_plain = (
                            f'{_n_unknown} epoch(s) ({_pct_unknown:.1f}%) have unrecognised label(s): '
                            f'{", ".join(_unknown_labels)}. Treated as artifact in the spectrogram.'
                        )
                except Exception as e:
                    hypno_warning = f'Error loading hypnogram: {e}'
                    hypno_warning_plain = f'Error loading hypnogram: {e}'

            # --- Check config ---
            if file_id not in config_dict:
                with out:
                    print(f'WARNING: {file_id} not in config JSON, skipping.')
                failed.append({'file_id': file_id, 'reason': 'not in config JSON'})
                continue

            sub_config = config_dict[file_id]
            # include= utilise les noms d'ORIGINE de l'EDF (les clés du remap), évalué à la
            # LECTURE. On ne garde ainsi que les canaux du montage : la fréquence native de
            # l'EEG est préservée (pas de suréchantillonnage vers un canal hors-montage plus
            # rapide comme un ECG 512 Hz), et on évite un AssertionError de lecture partielle
            # du lecteur EDF de MNE. Même motif que preprocessing_voila.
            selected_channels = list(sub_config['remap'].keys())

            # --- Load EDF ---
            try:
                raw = mne.io.read_raw_edf(
                    str(edf_path), preload=True, encoding='latin-1',
                    include=selected_channels, verbose=False
                )
            except Exception as e:
                with out:
                    print(f'ERROR loading {file_id}: {e}')
                failed.append({'file_id': file_id, 'reason': f'EDF loading: {e}'})
                continue

            # --- Save physical bounds before channel manipulation ---
            extras = raw._raw_extras[0]
            phys_bounds_by_name = {}
            for idx, ch in enumerate(raw.ch_names):
                base = ch.rsplit('-', 1)[0] if (ch.endswith('-0') or ch.endswith('-1')) else ch
                phys_bounds_by_name[base] = get_phys_bounds_uV(extras, idx)

            # --- Drop suffix duplicates, build remap, rename ---
            raw, _ = drop_suffix_duplicates(raw)
            remap_adapted = adapt_remap_dict_to_suffixes(raw, sub_config['remap'])
            final_to_orig = {}
            for orig, new in remap_adapted.items():
                base = orig.rsplit('-', 1)[0] if (orig.endswith('-0') or orig.endswith('-1')) else orig
                final_to_orig[new] = base
            raw.rename_channels(remap_adapted)
            sf = raw.info['sfreq']

            # --- Hypnogram length check ---
            if hypno_vec is not None:
                expected = int(np.floor(raw.n_times / sf / 30))
                if expected != len(hypno_vec):
                    hypno_warning = (
                        f'Length mismatch: EEG has {expected} 30s-epochs, '
                        f'hypnogram has {len(hypno_vec)}. Spectrogram skipped.'
                    )
                    hypno_warning_plain = (
                        f'Length mismatch: EEG has {expected} 30s-epochs, '
                        f'hypnogram has {len(hypno_vec)}. Spectrogram skipped.'
                    )
                    hypno_vec = None

            # --- Compute metrics for all channels (unfiltered signal) ---
            try:
                data_uV = raw.get_data() * 1e6
            except Exception as e:
                with out:
                    print(f'ERROR reading signal data for {file_id}: {e}')
                failed.append({'file_id': file_id, 'reason': f'signal data: {e}'})
                continue

            y_lim_ts = float(np.percentile(np.abs(data_uV), 99.9))
            # Shared X-limit for histograms: same p99.9 percentile but without the 500 µV cap,
            # so extreme-amplitude channels (movement bursts, artefacts) still show their full range.
            x_lim_hist = float(np.percentile(np.abs(data_uV), 99.9))

            ch_metrics = {}
            ch_flags = {}
            for ci, ch in enumerate(raw.ch_names):
                sig_uV = data_uV[ci]
                orig_base = final_to_orig.get(ch, ch)
                phys_min, phys_max = phys_bounds_by_name.get(orig_base, (np.nan, np.nan))
                m = compute_signal_metrics(sig_uV, phys_min, phys_max)
                ch_metrics[ch] = m
                ch_flags[ch] = flag_channel(m, thresholds)
                rows_summary.append({
                    'file_id': file_id,
                    'channel': ch,
                    'mean_uV': m['mean_uV'],
                    'std_uV': m['std_uV'],
                    'kurtosis': m['kurtosis'],
                    'skewness': m['skewness'],
                    'p99_abs_uV': m['p99_abs_uV'],
                    'p999_abs_uV': m['p999_abs_uV'],
                    'flat_pct': m['flat_pct'],
                    'bounds_pct': m['bounds_pct'],
                    'hist_extreme_pct': m['hist_extreme_pct'],
                    'n_peaks': m['n_peaks'],
                    'suspect_reason': '; '.join(ch_flags[ch]),
                    'exclude': bool(ch_flags[ch]),
                })

            # Shared Y-axis for histograms: computed from non-suspect channels only.
            # A flat or dead channel concentrates all samples in 1-2 bins, giving a
            # y_max that would crush the distributions of healthy channels.
            # Flagged channels are plotted with their own auto-scale (y_max=None) to
            # avoid their distribution being clipped by the healthy-channel scale.
            healthy_chs = [ch for ch, f in ch_flags.items() if not f]
            if healthy_chs:
                hist_y_max = max(float(np.max(ch_metrics[ch]['histo'])) for ch in healthy_chs)
            elif ch_metrics:
                hist_y_max = max(float(np.max(m['histo'])) for m in ch_metrics.values())
            else:
                hist_y_max = None

            # --- Build mne.Report ---
            report = mne.Report(title=f'{file_id} — Quality Overview', verbose=False)

            # Overview section: channel flag summary
            flagged = {ch: f for ch, f in ch_flags.items() if f}
            if flagged:
                items = ''.join(
                    f'<li><b>{ch}</b>: {", ".join(f)}</li>' for ch, f in flagged.items()
                )
                flag_html = (
                    '<div style="background:#f8d7da;padding:10px;'
                    'border-left:4px solid #dc3545;border-radius:4px;margin-bottom:12px;">'
                    f'<b>⚠ Suspicious channels ({len(flagged)}/{len(raw.ch_names)}):'
                    f'</b><ul>{items}</ul></div>'
                )
            else:
                flag_html = (
                    '<div style="background:#d4edda;padding:10px;'
                    'border-left:4px solid #28a745;border-radius:4px;margin-bottom:12px;">'
                    '<b>✓ All channels within thresholds</b></div>'
                )
            if hypno_warning:
                flag_html += (
                    '<div style="background:#fff3cd;padding:8px;'
                    'border-left:4px solid #ffc107;border-radius:4px;margin-bottom:12px;">'
                    f'⚠ {hypno_warning}</div>'
                )
            report.add_html(html=flag_html, title='Overview', section='Overview')

            # One section per channel: histogram + time series + spectrogram + metrics table
            for ci, ch in enumerate(raw.ch_names):
                m = ch_metrics[ch]
                flags = ch_flags[ch]
                sig_uV = data_uV[ci]

                # Flagged channels use their own auto-scale; healthy channels share hist_y_max.
                y_max_ch = hist_y_max if not flags else None
                fig = plot_histogram_figure(m, ch, y_max=y_max_ch, x_lim=x_lim_hist)
                report.add_figure(fig=fig, title=f'{ch} — histogram', section=ch,
                                  image_format='PNG')
                plt.close(fig)

                fig_ts = plot_timeseries_figure(sig_uV, sf, y_lim_ts)
                report.add_figure(fig=fig_ts, title=f'{ch} — time series', section=ch,
                                  image_format='PNG')
                plt.close(fig_ts)

                # Spectrogram: bandpass-filtered copy (0.1–40 Hz) computed just before plotting.
                if hypno_vec is not None:
                    try:
                        sig_filt = mne.filter.filter_data(
                            sig_uV[np.newaxis, :], sfreq=sf, l_freq=0.1, h_freq=40,
                            method='fir', phase='zero-double', fir_window='hamming', verbose=False
                        )[0]
                        hypno_full = yasa.hypno_upsample_to_data(
                            hypno=hypno_vec, sf_hypno=1 / 30, data=sig_filt, sf_data=sf
                        )
                        fig_spec = yasa.plot_spectrogram(sig_filt, sf, hypno_full, fmin=0.5, fmax=40)
                        fig_spec.set_size_inches(10, 3.5)
                        for ax in fig_spec.axes:
                            ax.tick_params(labelsize=8)
                            ax.xaxis.label.set_fontsize(11)
                            ax.yaxis.label.set_fontsize(11)
                        fig_spec.tight_layout()
                        report.add_figure(fig=fig_spec, title=f'{ch} — spectrogram', section=ch,
                                          image_format='PNG')
                        plt.close(fig_spec)
                    except Exception as _spec_err:
                        report.add_html(
                            html=f'<p>Spectrogram generation failed: {_spec_err}</p>',
                            title=f'{ch} — spectrogram', section=ch
                        )
                        with out:
                            print(f'  ⚠ {file_id}/{ch}: spectrogram error: {_spec_err}')
                else:
                    report.add_html(
                        html='<p>No spectrogram: hypnogram missing or incompatible with EEG length.</p>',
                        title=f'{ch} — spectrogram', section=ch
                    )

                flag_bg = '#f8d7da' if flags else '#f0f0f0'
                flag_text = '; '.join(flags) if flags else 'no flags'
                rows_html = ''.join([
                    f'<tr><td style="padding:4px 10px;">{label}</td>'
                    f'<td style="padding:4px 10px;text-align:right;">{val}</td>'
                    f'<td style="padding:4px 10px;color:#555;font-size:0.85em;">'
                    f'{interpretations.get(label, "")}</td></tr>'
                    for label, val in [
                        ('Mean', f"{m['mean_uV']:.2f} µV"),
                        ('Std dev', f"{m['std_uV']:.2f} µV"),
                        ('p99 |amplitude|', f"{m['p99_abs_uV']:.1f} µV"),
                        ('p99.9 |amplitude|', f"{m['p999_abs_uV']:.1f} µV"),
                        ('Kurtosis (Fischer)', f"{m['kurtosis']:.2f}"),
                        ('Skewness', f"{m['skewness']:.3f}"),
                        ('Flat signal (%)', f"{m['flat_pct']:.2f}%"),
                        ('At EDF bounds (%)', f"{m['bounds_pct']:.3f}%"),
                        ('Extreme histogram (%)', f"{m['hist_extreme_pct']:.3f}%"),
                        ('n_peaks (distribution)', str(m['n_peaks'])),
                    ]
                ])
                metrics_html = (
                    f'<div style="background:{flag_bg};padding:8px;border-radius:4px;'
                    f'margin-bottom:8px;"><b>Flags:</b> {flag_text}</div>'
                    '<table style="border-collapse:collapse;font-size:0.9em;width:100%;">'
                    '<tr style="background:#eee;">'
                    '<th style="padding:4px 10px;text-align:left;">Metric</th>'
                    '<th style="padding:4px 10px;text-align:right;">Value</th>'
                    '<th style="padding:4px 10px;text-align:left;">Interpretation</th></tr>'
                    f'{rows_html}</table>'
                )
                report.add_html(html=metrics_html, title=f'{ch} — metrics', section=ch)

            report.save(str(report_path), overwrite=True, open_browser=False, verbose=False)
            n_flagged = sum(1 for f in ch_flags.values() if f)
            with out:
                print(f'{file_id}: {n_flagged}/{len(raw.ch_names)} channel(s) flagged  →  {report_path.name}')
                if hypno_warning_plain:
                    print(f'  ⚠ {hypno_warning_plain}')

        # --- Save / merge global TSV files ---
        # TSV outputs are always written to reports_root (the root output folder),
        # regardless of subfolder structure, to give a single aggregated dataset view.
        # Any file_id that was attempted (not skipped) replaces its previous entry.
        df_new = pd.DataFrame(rows_summary)
        summary_path = reports_root / 'quality_summary.tsv'

        if not df_new.empty:
            if summary_path.exists() and attempted_ids:
                df_existing = pd.read_csv(str(summary_path), sep='\t')
                df_existing = df_existing[~df_existing['file_id'].isin(attempted_ids)]
                df_to_save = pd.concat([df_existing, df_new], ignore_index=True)
            else:
                df_to_save = df_new
            df_to_save.to_csv(str(summary_path), sep='\t', index=False)

        # Regenerate dataset_overview.html from the full cumulative quality_summary.tsv
        # so the overview reflects all participants processed to date, not just this run.
        if summary_path.exists():
            try:
                generate_dataset_overview(
                    pd.read_csv(str(summary_path), sep='\t'), reports_root
                )
            except Exception as _ov_err:
                with out:
                    print(f'⚠ dataset_overview.html generation failed: {_ov_err}')

        failed_path = reports_root / 'failed_files.tsv'
        if failed:
            df_failed_new = pd.DataFrame(failed)
            if failed_path.exists() and attempted_ids:
                df_failed_existing = pd.read_csv(str(failed_path), sep='\t')
                df_failed_existing = df_failed_existing[~df_failed_existing['file_id'].isin(attempted_ids)]
                df_failed = pd.concat([df_failed_existing, df_failed_new], ignore_index=True)
            else:
                df_failed = df_failed_new
            df_failed.to_csv(str(failed_path), sep='\t', index=False)
        elif failed_path.exists() and attempted_ids:
            # A file that was previously failing and was re-attempted successfully: remove its entry.
            df_failed_existing = pd.read_csv(str(failed_path), sep='\t')
            df_failed_cleaned = df_failed_existing[~df_failed_existing['file_id'].isin(attempted_ids)]
            if df_failed_cleaned.empty:
                failed_path.unlink()
            elif len(df_failed_cleaned) < len(df_failed_existing):
                df_failed_cleaned.to_csv(str(failed_path), sep='\t', index=False)

        progress.value = len(edf_files)
        progress_label.value = 'Done.'

        elapsed = time.time() - t_start
        mins, secs = divmod(elapsed, 60)
        elapsed_str = f'{int(mins)} min {secs:.0f} s' if mins >= 1 else f'{secs:.1f} s'

        with out:
            if not df_new.empty:
                n_files_done = int(df_new['file_id'].nunique())
                n_suspect_participants = int(df_new.groupby('file_id')['exclude'].any().sum())
                n_suspect = int(df_new['exclude'].sum())
            else:
                n_files_done = n_suspect_participants = n_suspect = 0
            avg_str = f'{elapsed / n_files_done:.0f} s / participant' if n_files_done > 0 else 'N/A'
            print(f'\n=== Analysis complete ===')
            print(f'Output folder              : {reports_root}')
            if (reports_root / 'dataset_overview.html').exists():
                print(f'Dataset overview           : dataset_overview.html')
            print(f'Participants processed     : {n_files_done}')
            print(f'Participants with ≥1 flag  : {n_suspect_participants} / {n_files_done}')
            print(f'Total flagged channels     : {n_suspect}')
            if failed:
                print(f'Files failed to load       : {len(failed)}')
            print(f'Total time                 : {elapsed_str}')
            print(f'Average per participant    : {avg_str}')

    except Exception as _run_err:
        with out:
            print(f'\nERREUR inattendue — analyse interrompue : {_run_err}')
    finally:
        btn.disabled = False


btn_run.on_click(run_analysis)

display(
    HTML('<h3 style="margin-top:24px;">&#9654; Run analysis</h3>'),
    btn_run,
    widgets.HBox([progress, progress_label]),
    out
)